In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Assignment — Linear Regression on penguins.csv
Question: given a penguin's measurements, how heavy is it?

Target: body_mass_g  ·  File: penguins.csv

Use only the numeric columns: bill_length_mm, bill_depth_mm, flipper_length_mm.

In [4]:
penguin=pd.read_csv("./penguins.csv")

#### 1. Look at the data
Flow: Shape, columns, missing values.

Run .info(). Which columns have missing values, and how many?

In [5]:
penguin.info()
print("Shape:",penguin.shape)

print("Missing data points count: ", penguin.isna().sum().sum())

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    str    
dtypes: float64(4), str(3)
memory usage: 18.9 KB
Shape: (344, 7)
Missing data points count:  19


#### 2. Clean
Flow: Keep the four numeric columns, drop the rows with blanks.

We cannot fill in body_mass_g — that is the answer we are trying to predict. Rows missing it have to go.

Keep bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g, then dropna().

In [6]:
penguin.drop(columns=['species', 'island', 'sex'], inplace=True)

In [7]:
penguin.info()

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   bill_length_mm     342 non-null    float64
 1   bill_depth_mm      342 non-null    float64
 2   flipper_length_mm  342 non-null    float64
 3   body_mass_g        342 non-null    float64
dtypes: float64(4)
memory usage: 10.9 KB


In [ ]:
penguin.dropna(subset=['body_mass_g'], inplace=True)

In [10]:
penguin.info()

<class 'pandas.DataFrame'>
Index: 342 entries, 0 to 343
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   bill_length_mm     342 non-null    float64
 1   bill_depth_mm      342 non-null    float64
 2   flipper_length_mm  342 non-null    float64
 3   body_mass_g        342 non-null    float64
dtypes: float64(4)
memory usage: 13.4 KB


#### 3. Features and target

In [11]:
y=penguin['body_mass_g'];
x=penguin.drop(columns=['body_mass_g'])

In [13]:
print(x.head())

   bill_length_mm  bill_depth_mm  flipper_length_mm
0            39.1           18.7              181.0
1            39.5           17.4              186.0
2            40.3           18.0              195.0
4            36.7           19.3              193.0
5            39.3           20.6              190.0


In [14]:
print(y.head())

0    3750.0
1    3800.0
2    3250.0
4    3450.0
5    3650.0
Name: body_mass_g, dtype: float64


#### 4. Split
Flow: test_size=0.2, random_state=42.

In [15]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2, random_state=42)

In [17]:
print(x_train.head())

     bill_length_mm  bill_depth_mm  flipper_length_mm
115            42.7           18.3              196.0
8              34.1           18.1              193.0
138            37.0           16.5              185.0
333            51.5           16.3              230.0
305            50.8           17.3              228.0


In [18]:
print(y_train.head())

115    4075.0
8      3475.0
138    3400.0
333    5500.0
305    5600.0
Name: body_mass_g, dtype: float64


In [19]:
print(x_test.head())

     bill_length_mm  bill_depth_mm  flipper_length_mm
238            46.2           14.5              209.0
117            37.3           20.5              199.0
114            39.6           20.7              191.0
43             44.1           19.7              196.0
127            41.5           18.3              195.0


In [20]:
print(y_test.head())

238    4800.0
117    3775.0
114    3900.0
43     4400.0
127    4300.0
Name: body_mass_g, dtype: float64


#### 5. One feature
Flow: Start with flipper_length_mm alone.

Train LinearRegression. Print the coefficient, the intercept and the R².

In [21]:
from sklearn.linear_model import LinearRegression
model=LinearRegression()

In [23]:
model.fit(x_train[['flipper_length_mm']], y_train)
print("Coefficient:",model.coef_)
print("Intercept:",model.intercept_)
print("R2:",model.score(x_test[['flipper_length_mm']], y_test))

Coefficient: [48.82968282]
Intercept: -5614.067120606903
R2: 0.7820354165340793


Q. The coefficient is about 48. Write one line explaining what that means in plain words — what happens to the predicted weight if a penguin's flipper is 1 mm longer?

Ans: A coefficient of about 48 means that for every 1 mm increase in a penguin's flipper length, the model predicts that its body mass will increase by approximately 48 grams.

#### 6. All three features
Flow: Same model, two more columns.

Print the R².

In [24]:
model.fit(x_train, y_train)
print("R2", model.score(x_test,y_test))

R2 0.7877806019338434


R2 with flipper length: 0.7820354165340793; 
R2 with all features: 0.7877806019338434

Q. The score barely moved. That is not a mistake — it is the interesting part of this assignment. Write two lines on why adding bill_length_mm and bill_depth_mm gave almost nothing.



Ans: Adding the bill measurements barely improves the score because all three columns are highly correlated proxies for the penguin's overall size. Once flipper length already accounts for how large the penguin is, the bill dimensions offer almost no new, non-redundant information about its weight.

#### 7. Metrics
Flow: MAE, RMSE, R².

In [27]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
model.fit(x_train, y_train)
y_predict=model.predict(x_test)

print("Mean absolute error", mean_absolute_error(y_predict,y_test))
print("Root mean squared error", np.sqrt(mean_squared_error(y_predict,y_test)))
print("R2 score", r2_score(y_predict, y_test))

Mean absolute error 310.54744049126737
Root mean squared error 375.64413431107766
R2 score 0.6736342286197672


Q. The average penguin weighs about 4200 g. Is your MAE large or small compared to that? One line.

Ans: MAE is smaller than avg penguin weight

#### 8. Coefficients
Print each feature name next to its coefficient

In [28]:
x_train.head(1)

,bill_length_mm,bill_depth_mm,flipper_length_mm
115,42.7,18.3,196.0


In [29]:
for col in ('bill_length_mm', 'bill_depth_mm', 'flipper_length_mm'):
    model=LinearRegression()
    model.fit(x_train[[col]], y_train)
    print(col, model.coef_)

bill_length_mm [85.05637403]
bill_depth_mm [-190.86518255]
flipper_length_mm [48.82968282]


Largest coefficient: bill_length_mm

Q. Does the largest coefficient mean that feature is the most important? Careful — the three columns are measured on different scales. One line.

Ans: No, because the data points are not standarized

#### 9. Questions
Q1. Why can we not use accuracy_score on this problem?

Q2. We dropped rows where body_mass_g was missing instead of filling them with the median. Why is filling in the target a bad idea?

Q3. In class, adding more features to the mpg model raised R² from 0.723 to 0.824. Here it barely changed. In two lines — what is different about these two datasets?

##### Answers:
1. Accuracy_score is measured in case of classification. This is a question fro regression case.

2. We are predicting the body_mass_g through our model. It may occur that filling them with median may affect the prediction and the model may be unefficient. Hence, we prevent the filling of data for the output column

3. In mpg, the mpg also was dependent on other factors which gave different information helping in prediction of the mpg. Here, all the factors gives the idea of the body dimension of the penguin and generally no other important information that would affect the body mass. Thus, here increasing the columns did not affect the R2 scores so much.